In [ ]:
import copy
import json
import math
import time
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
from datasets import load_dataset
from huggingface_hub import snapshot_download
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoTokenizer, Qwen3Model, get_cosine_schedule_with_warmup

# =========================================================
# 1. CONFIGURATION & PATHS
# =========================================================
HF_REPO_ID = "Piyush-5541/bev-decision-lora"
HF_TOKEN = None  # Set to "hf_..." if your repo is private

# Point these to your 30k train file and ~6k test file
TRAIN_JSONL_PATH = "/content/drive/MyDrive/rlcb/train_12000.jsonl"
TEST_JSONL_PATH = "/content/drive/MyDrive/rlcb/test_4000.jsonl"

RLCB_CONFIG = {
    "seed": 42,
    "data": {
        "rlcb_train_samples": 12000,  
        "rlcb_test_samples": 4000,    
        "batch_size": 4,
        "num_workers": 0,
    },
    "optim": {
        "epochs": 1,
        "max_steps": 600,             # 600 steps * batch_size 4 * accum 2 = 4,800 calibrated decisions
        "grad_accum_steps": 2,
        "lr_lora": 2e-5,              # Gentle LR so we don't destroy SFT representations
        "lr_head": 1e-4,
        "weight_decay": 0.01,
        "warmup_ratio": 0.05,
        "grad_clip": 1.0,
    },
    "rlcb": {
        "kl_beta": 0.04,              # KL divergence anchor weight to your SFT checkpoint
        "pg_weight": 0.3,             # Contextual bandit policy-gradient weight (Leave-One-Out)
        "num_samples_M": 4,           # Number of bandit rollouts per question
    },
    "logging": {
        "log_every": 25,
        "eval_every": 200,
    },
}

TASK_TYPES = {"choice": 0, "noul": 1, "score": 2}
TASK_NAMES = {v: k for k, v in TASK_TYPES.items()}
PROMPTS = {
    "choice": "Choose the best option.",
    "score": "Rate it on the given scale.",
    "noul": "Answer yes or no.",
}

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else "cpu"
)
torch.manual_seed(RLCB_CONFIG["seed"])


def amp():
    if DEVICE.type == "cuda":
        return torch.autocast("cuda", dtype=torch.bfloat16)
    if DEVICE.type == "mps":
        return torch.autocast("mps", dtype=torch.float16)
    return torch.autocast("cpu", enabled=False)

In [ ]:
# =========================================================
# 2. BEV MODEL ARCHITECTURE (Exact Match to Your Notebook)
# =========================================================
class SA(nn.Module):
    def __init__(self, h, d):
        super().__init__()
        self.d = d
        self.q = nn.Linear(h, d)
        self.k = nn.Linear(h, d)
        self.v = nn.Linear(h, d)
        self.o = nn.Linear(d, h)

    def forward(self, x, valid):
        scores = torch.einsum("bqd,bkd->bqk", self.q(x), self.k(x)) / math.sqrt(self.d)
        weights = torch.softmax(scores.masked_fill(~valid[:, None, :], -float("inf")), dim=-1)
        return self.o(torch.einsum("bqk,bkd->bqd", weights, self.v(x)))


class Block(nn.Module):
    def __init__(self, h, d):
        super().__init__()
        self.n1 = nn.LayerNorm(h)
        self.a = SA(h, d)
        self.n2 = nn.LayerNorm(h)
        self.m = nn.Sequential(nn.Linear(h, d), nn.GELU(), nn.Linear(d, h))

    def forward(self, x, v):
        x = x + self.a(self.n1(x), v)
        return x + self.m(self.n2(x))


class Head(nn.Module):
    def __init__(self, h, new_dim, num_layers, num_task_types):
        super().__init__()
        self.d = new_dim
        self.task_embedding = nn.Embedding(num_task_types, h)
        self.norm = nn.LayerNorm(h)
        self.blocks = nn.ModuleList([Block(h, new_dim) for _ in range(num_layers)])
        self.final = nn.LayerNorm(h)
        self.ap = nn.Linear(h, new_dim)
        self.cp = nn.Linear(h, new_dim)

    def forward(self, choices, answer, choice_mask, task):
        x = self.norm(
            torch.cat([self.task_embedding(task).unsqueeze(1), choices, answer.unsqueeze(1)], 1)
        )
        one = torch.ones(len(x), 1, dtype=torch.bool, device=x.device)
        valid = torch.cat([one, choice_mask, one], 1)
        for block in self.blocks:
            x = block(x, valid)
        x = self.final(x)
        logits = torch.einsum("bd,bcd->bc", self.ap(x[:, -1]), self.cp(x[:, 1:-1])) / math.sqrt(self.d)
        return logits.masked_fill(~choice_mask, -float("inf"))


class Network(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, input_ids, position_ids, attention_mask, choice_read_idx, choice_mask, answer_end_idx, task_type, **_):
        h = self.backbone(
            input_ids=input_ids,
            position_ids=position_ids,
            attention_mask=attention_mask,
            use_cache=False,
        ).last_hidden_state.float()
        b = torch.arange(len(h), device=h.device)
        with torch.autocast(h.device.type, enabled=False):
            return self.head(h[b[:, None], choice_read_idx], h[b, answer_end_idx], choice_mask, task_type)

In [ ]:
# =========================================================
# 3. LOAD TRAINED MODEL FROM HUGGING FACE & COUNT PARAMS
# =========================================================
def load_sft_from_hub(repo_id, token=None):
    print(f"Fetching trained SFT checkpoint from Hugging Face: {repo_id}...")
    folder = Path(snapshot_download(repo_id=repo_id, token=token))
    meta = json.loads((folder / "meta.json").read_text())
    tok = AutoTokenizer.from_pretrained(meta["model_name"])

    base = Qwen3Model.from_pretrained(
        meta["model_name"], dtype=torch.float32, num_hidden_layers=meta["num_layers"]
    )
    # Load your trained LoRA adapter as "default" (trainable) and a frozen copy as "sft_ref"
    base = PeftModel.from_pretrained(base, folder / "lora", adapter_name="default", is_trainable=True)
    base.load_adapter(str(folder / "lora"), adapter_name="sft_ref", is_trainable=False)
    base.set_adapter("default")

    head = Head(base.config.hidden_size, **meta["head_config"])
    head.load_state_dict(torch.load(folder / "head.pt", map_location="cpu"))

    # Frozen reference head for KL anchor (only 6.3M params)
    ref_head = copy.deepcopy(head).to(DEVICE).eval()
    ref_head.requires_grad_(False)

    net = Network(base, head).to(DEVICE)
    return net, ref_head, tok, meta


def print_parameter_breakdown(net):
    # Count only the active ("default") adapter + base + head
    base_frozen = sum(p.numel() for n, p in net.backbone.named_parameters() if "lora_" not in n)
    lora_trainable = sum(p.numel() for n, p in net.backbone.named_parameters() if "lora_" in n and "default" in n)
    head_trainable = sum(p.numel() for p in net.head.parameters())
    total_active = base_frozen + lora_trainable + head_trainable
    total_trainable = lora_trainable + head_trainable

    print("\n" + "=" * 62)
    print("MODEL PARAMETER BREAKDOWN (20-Layer Qwen3-0.6B + LoRA + Head)")
    print("=" * 62)
    print(f"  Frozen Qwen3-0.6B Backbone (20/28 layers) : {base_frozen:>13,} params")
    print(f"  Trainable LoRA Adapters (Blocks 16-19)    : {lora_trainable:>13,} params")
    print(f"  Trainable Custom Decision Head (2 blocks) : {head_trainable:>13,} params")
    print("-" * 62)
    print(f"  TOTAL MODEL PARAMETERS (Active)           : {total_active:>13,} (~{total_active/1e6:.2f}M)")
    print(f"  TOTAL TRAINABLE PARAMETERS                : {total_trainable:>13,} (~{total_trainable/1e6:.2f}M | {100*total_trainable/total_active:.2f}%)")
    print("=" * 62 + "\n")

In [ ]:
# =========================================================
# 4. DATASET LOADING & TOKENIZATION
# =========================================================
def build_dataloader(jsonl_path, tok, meta, max_questions, batch_size, shuffle, seed):
    def trunc(text, maximum):
        ids = tok.encode(text)
        return text if len(ids) <= maximum else tok.decode(ids[:maximum])

    def choices_for(q):
        if q["type"] == "choice":
            keys = list(q["criteria"])
            return [f"{k}: {q['criteria'][k]}" if q["criteria"][k] else k for k in keys], keys.index(q["label"]) if "label" in q else None
        if q["type"] == "score":
            return [f"{i}: {x}" for i, x in enumerate(q["criteria"])], q.get("label")
        return ["No", "Yes"], int(q["label"]) if "label" in q else None

    def expand(rows):
        out = {k: [] for k in ("state", "task_type", "instructions", "choices", "label")}
        for state, raw in zip(rows["state"], rows["questions_json"]):
            for q in json.loads(raw).values():
                options, label = choices_for(q)
                if label is None:
                    continue
                out["state"].append(state)
                out["task_type"].append(TASK_TYPES[q["type"]])
                out["instructions"].append(q["instructions"])
                out["choices"].append(options)
                out["label"].append(label)
        return out

    ds = load_dataset("json", data_files=str(jsonl_path), split="train").shuffle(seed=seed)
    if max_questions is not None:
        ds = ds.select(range(min(max_questions, len(ds))))
    ds = ds.map(expand, batched=True, remove_columns=ds.column_names)
    if max_questions is not None:
        ds = ds.select(range(min(max_questions, len(ds))))

    def encode(state, instruction, options, kind):
        question = f"{PROMPTS[kind]}\nQuestion: {instruction}\n\n{trunc(state, meta['max_state_tokens'])}\n"
        tokens = tok.encode(question)
        q_end = len(tokens) - 1
        ends = []
        begin, close = tok.encode("<option>"), tok.encode("</option>")
        for option in options:
            tokens += begin + tok.encode(trunc(option, meta["max_choice_tokens"])) + close
            ends.append(len(tokens) - 1)
        tokens += tok.encode("The answer is:")
        answer_end = len(tokens) - 1

        positions = list(range(q_end + 1))
        base = len(positions)
        next_start = base
        for end in ends:
            positions += list(range(base, base + end - next_start + 1))
            next_start = end + 1
        positions += list(range(max(positions) + 1, max(positions) + 1 + answer_end + 1 - len(positions)))

        mask = torch.zeros((len(tokens), len(tokens)))
        q_len = q_end + 1
        mask[:q_len, :q_len] = torch.tril(torch.ones(q_len, q_len))
        start = q_len
        for end in ends:
            width = end - start + 1
            mask[start : end + 1, start : end + 1] = torch.tril(torch.ones(width, width))
            start = end + 1
        mask[q_len:, :q_len] = 1
        mask[start:, :start] = 1
        mask[start:, start:] = torch.tril(torch.ones(len(tokens) - start, len(tokens) - start))
        return {
            "input_ids": tokens,
            "position_ids": positions,
            "attention_mask": mask,
            "choice_read_idx": [x - len(close) for x in ends],
            "answer_end_idx": answer_end,
        }

    class BEVDataset(torch.utils.data.Dataset):
        def __init__(self, rows):
            self.rows = rows
        def __len__(self):
            return len(self.rows)
        def __getitem__(self, i):
            r = self.rows[i]
            x = encode(r["state"], r["instructions"], r["choices"], TASK_NAMES[r["task_type"]])
            x.update(task_type=r["task_type"], label=r["label"])
            return x

    def collate(batch):
        b = len(batch)
        length = max(len(x["input_ids"]) for x in batch)
        count = max(len(x["choice_read_idx"]) for x in batch)
        ids = torch.full((b, length), tok.pad_token_id, dtype=torch.long)
        pos = torch.zeros((b, length), dtype=torch.long)
        mask = torch.eye(length).repeat(b, 1, 1)
        read = torch.zeros((b, count), dtype=torch.long)
        valid = torch.zeros((b, count), dtype=torch.bool)
        answer = torch.zeros(b, dtype=torch.long)
        for i, x in enumerate(batch):
            n, c = len(x["input_ids"]), len(x["choice_read_idx"])
            ids[i, :n] = torch.tensor(x["input_ids"])
            pos[i, :n] = torch.tensor(x["position_ids"])
            mask[i, :n, :n] = x["attention_mask"]
            read[i, :c] = torch.tensor(x["choice_read_idx"])
            valid[i, :c] = True
            answer[i] = x["answer_end_idx"]
        return {
            "input_ids": ids,
            "position_ids": pos,
            "attention_mask": torch.where(mask == 0, -float("inf"), 0.0).unsqueeze(1),
            "choice_read_idx": read,
            "choice_mask": valid,
            "answer_end_idx": answer,
            "task_type": torch.tensor([x["task_type"] for x in batch]),
            "labels": torch.tensor([x["label"] for x in batch]),
        }

    return torch.utils.data.DataLoader(
        BEVDataset(ds),
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collate,
        num_workers=RLCB_CONFIG["data"]["num_workers"],
        pin_memory=(DEVICE.type == "cuda"),
    )


# =========================================================
# 5. RLCB / RLCD LOSS & CALIBRATION EVALUATION
# =========================================================
@torch.no_grad()
def get_ref_logits(net, ref_head, batch):
    """Computes reference logits from your original SFT checkpoint without duplicating Qwen in VRAM."""
    net.backbone.set_adapter("sft_ref")
    net.backbone.eval()
    h = net.backbone(
        input_ids=batch["input_ids"],
        position_ids=batch["position_ids"],
        attention_mask=batch["attention_mask"],
        use_cache=False,
    ).last_hidden_state.float()
    b = torch.arange(len(h), device=h.device)
    with torch.autocast(h.device.type, enabled=False):
        ref_logits = ref_head(h[b[:, None], batch["choice_read_idx"]], h[b, batch["answer_end_idx"]], batch["choice_mask"], batch["task_type"])
    net.backbone.set_adapter("default")
    net.backbone.train()
    return ref_logits


def rlcb_loss_fn(logits, ref_logits, labels, choice_mask, rl_cfg):
    """
    Combines:
      1. Strictly Proper Brier Score: R(u, y) = 2*u_y - sum(u_k^2)
      2. KL Anchor to SFT reference model: beta * KL(u || u_ref)
      3. Multi-sample Leave-One-Out (LOO) Contextual Bandit Policy Gradient
    """
    u = F.softmax(logits, dim=-1) * choice_mask.float()
    log_u = F.log_softmax(logits, dim=-1).clamp(min=-1e4)

    with torch.no_grad():
        u_ref = F.softmax(ref_logits, dim=-1) * choice_mask.float()
        log_u_ref = F.log_softmax(ref_logits, dim=-1).clamp(min=-1e4)

    # 1. Brier Score (higher = better calibrated + accurate, max = 1.0)
    u_y = u.gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    brier_reward = 2.0 * u_y - (u ** 2).sum(dim=-1)

    # 2. KL Divergence to SFT Checkpoint
    kl_div = (u * (log_u - log_u_ref) * choice_mask.float()).sum(dim=-1)

    # Stage 1: Pathwise proper-scoring objective
    pathwise_loss = -(brier_reward - rl_cfg["kl_beta"] * kl_div).mean()

    # Stage 2: Leave-One-Out (LOO) Contextual Bandit Policy Gradient
    M = rl_cfg["num_samples_M"]
    sampled_a = torch.multinomial(u.clamp(min=1e-8), num_samples=M, replacement=True)
    log_pi_a = log_u.gather(-1, sampled_a)
    log_pi_ref_a = log_u_ref.gather(-1, sampled_a)

    with torch.no_grad():
        hit = (sampled_a == labels.unsqueeze(-1)).float()
        R_m = hit - rl_cfg["kl_beta"] * (log_pi_a - log_pi_ref_a)
        loo_baseline = (R_m.sum(dim=-1, keepdim=True) - R_m) / max(1, M - 1)
        advantage = (R_m - loo_baseline) / M

    bandit_loss = -(advantage.detach() * log_pi_a).sum(dim=-1).mean()
    total_loss = pathwise_loss + rl_cfg["pg_weight"] * bandit_loss

    return total_loss, brier_reward.mean().item(), kl_div.mean().item()


@torch.no_grad()
def evaluate_calibration(net, test_loader):
    """Evaluates Accuracy, Brier Error, Cross-Entropy, and Expected Calibration Error (ECE)."""
    net.eval()
    total_nll = 0.0
    total_brier_err = 0.0
    hits = defaultdict(int)
    totals = defaultdict(int)
    all_confs, all_correct = [], []

    for batch in tqdm(test_loader, desc="Evaluating Test Split", leave=False):
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        with amp():
            logits = net(**batch)

        probs = F.softmax(logits, dim=-1) * batch["choice_mask"].float()
        total_nll += F.cross_entropy(logits, batch["labels"], reduction="sum").item()

        # Brier Error: ||u - e_y||^2 (Lower is better, 0.0 is perfect)
        u_y = probs.gather(-1, batch["labels"].unsqueeze(-1)).squeeze(-1)
        brier_err = (1.0 - 2.0 * u_y + (probs ** 2).sum(dim=-1)).sum().item()
        total_brier_err += brier_err

        confs, preds = probs.max(dim=-1)
        corrects = (preds == batch["labels"])
        all_confs.append(confs.cpu())
        all_correct.append(corrects.float().cpu())

        for typ, hit in zip(batch["task_type"].tolist(), corrects.tolist()):
            name = TASK_NAMES[typ]
            totals[name] += 1
            hits[name] += int(hit)

    # Expected Calibration Error (10-bin ECE)
    confs_t = torch.cat(all_confs)
    corr_t = torch.cat(all_correct)
    ece = 0.0
    for b_low, b_high in zip(torch.linspace(0, 0.9, 10), torch.linspace(0.1, 1.0, 10)):
        in_bin = (confs_t > b_low) & (confs_t <= b_high)
        if in_bin.any():
            prop = in_bin.float().mean().item()
            ece += abs(corr_t[in_bin].mean().item() - confs_t[in_bin].mean().item()) * prop

    n = sum(totals.values())
    net.train()
    return {
        "accuracy": sum(hits.values()) / n,
        "brier_error": total_brier_err / n,
        "ece": ece,
        "nll_loss": total_nll / n,
        **{f"accuracy_{k}": hits[k] / totals[k] for k in totals},
    }


def save_rlcb_checkpoint(net, folder, meta, cfg, step):
    folder.mkdir(parents=True, exist_ok=True)
    # Save only the active ("default") adapter so directory structure matches your original repo
    net.backbone.save_pretrained(folder, selected_adapters=["default"])
    lora_src = folder / "default"
    lora_dst = folder / "lora"
    if lora_src.exists():
        if lora_dst.exists():
            import shutil
            shutil.rmtree(lora_dst)
        lora_src.rename(lora_dst)

    torch.save({k: v.cpu() for k, v in net.head.state_dict().items()}, folder / "head.pt")
    (folder / "rlcb_config.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
    meta_out = {**meta, "rlcb_step": step}
    (folder / "meta.json").write_text(json.dumps(meta_out, indent=2))

In [ ]:
!pip install --upgrade torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =========================================================
# 6. MAIN EXECUTION PIPELINE
# =========================================================
if __name__ == "__main__":
    run_dir = Path.cwd() / "runs" / f"rlcb-{datetime.now():%Y%m%d-%H%M%S}"
    run_dir.mkdir(parents=True, exist_ok=True)

    # 1. Load your SFT checkpoint from Hugging Face & print parameter summary
    net, ref_head, tok, meta = load_sft_from_hub(HF_REPO_ID, token=HF_TOKEN)
    print_parameter_breakdown(net)

    # 2. Load 5,000 train samples (from your 30k file) and ~6,000 test samples
    print("Building DataLoaders...")
    train_loader = build_dataloader(
        TRAIN_JSONL_PATH, tok, meta,
        max_questions=RLCB_CONFIG["data"]["rlcb_train_samples"],
        batch_size=RLCB_CONFIG["data"]["batch_size"],
        shuffle=True,
        seed=RLCB_CONFIG["seed"],
    )
    test_loader = build_dataloader(
        TEST_JSONL_PATH, tok, meta,
        max_questions=RLCB_CONFIG["data"]["rlcb_test_samples"],
        batch_size=RLCB_CONFIG["data"]["batch_size"],
        shuffle=False,
        seed=RLCB_CONFIG["seed"],
    )

    # 3. Baseline Evaluation (Before RLCB)
    print("\n--- Baseline Evaluation (Your HF SFT Model Before RLCB) ---")
    base_metrics = evaluate_calibration(net, test_loader)
    print(json.dumps(base_metrics, indent=2))

    # 4. Optimizer & Scheduler (Only trainable "default" LoRA + Head)
    lora_params = [p for n, p in net.backbone.named_parameters() if p.requires_grad and "default" in n]
    for n, p in net.backbone.named_parameters():
        if "sft_ref" in n:
            p.requires_grad_(False)

    opt = torch.optim.AdamW(
        [
            {"params": lora_params, "lr": RLCB_CONFIG["optim"]["lr_lora"]},
            {"params": net.head.parameters(), "lr": RLCB_CONFIG["optim"]["lr_head"]},
        ],
        weight_decay=RLCB_CONFIG["optim"]["weight_decay"],
    )

    accum = RLCB_CONFIG["optim"]["grad_accum_steps"]
    total_steps = min(RLCB_CONFIG["optim"]["max_steps"], len(train_loader) // accum)
    sched = get_cosine_schedule_with_warmup(
        opt, int(RLCB_CONFIG["optim"]["warmup_ratio"] * total_steps), total_steps
    )

    # 5. RLCB Training Loop
    print(f"\n--- Starting RLCB Post-Training for {total_steps} steps ---")
    step = 0
    best_brier = base_metrics["brier_error"]
    opt.zero_grad()

    with (run_dir / "rlcb_metrics.jsonl").open("a") as log_file:
        log_file.write(json.dumps({"step": 0, "split": "test_baseline", **base_metrics}) + "\n")

        pbar = tqdm(enumerate(train_loader, 1), total=total_steps * accum, desc="RLCB Training")
        for micro, batch in pbar:
            if step >= total_steps:
                break
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

            ref_logits = get_ref_logits(net, ref_head, batch)
            with amp():
                logits = net(**batch)
                loss, brier_rew, kl_val = rlcb_loss_fn(
                    logits, ref_logits, batch["labels"], batch["choice_mask"], RLCB_CONFIG["rlcb"]
                )

            (loss / accum).backward()

            if micro % accum != 0:
                continue

            torch.nn.utils.clip_grad_norm_(net.parameters(), RLCB_CONFIG["optim"]["grad_clip"])
            opt.step()
            sched.step()
            opt.zero_grad()
            step += 1

            pbar.set_postfix(step=step, loss=f"{loss.item():.4f}", brier_rew=f"{brier_rew:.3f}", kl=f"{kl_val:.4f}")

            if step % RLCB_CONFIG["logging"]["eval_every"] == 0:
                metrics = evaluate_calibration(net, test_loader)
                record = {"step": step, "split": "test", **metrics}
                log_file.write(json.dumps(record) + "\n")
                print(f"\n[Step {step}] Test Metrics:", record)
                # Save best checkpoint by lowest Brier Error (best calibrated accuracy)
                if metrics["brier_error"] < best_brier:
                    best_brier = metrics["brier_error"]
                    save_rlcb_checkpoint(net, run_dir / "checkpoints" / "best", meta, RLCB_CONFIG, step)

    # 6. Final Evaluation & Save
    final_metrics = evaluate_calibration(net, test_loader)
    save_rlcb_checkpoint(net, run_dir / "checkpoints" / "final", meta, RLCB_CONFIG, step)
    print("\n--- Final RLCB Test Metrics ---")
    print(json.dumps(final_metrics, indent=2))
    print(f"Saved RLCB checkpoints to: {run_dir / 'checkpoints'}")